# Error Analysis


## Setup & Imports

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"          # global (frozen prompts)

# UNI-88: point at one experiment folder. Paste the name printed by pipeline.ipynb §2.
EXPERIMENT_NAME = "experiment_test_9385_20260605_1335"       # <-- set to the run you are analysing
EXPERIMENT_DIR  = ROOT / "data" / "experiments" / EXPERIMENT_NAME
LLAMA_RUNS  = EXPERIMENT_DIR / "llama_runs"
EXPERIMENT  = EXPERIMENT_DIR / "experiment.jsonl"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
print(f"ROOT = {ROOT}")

ROOT = /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis


## (i) Lexical Bias

**Goal:** Retrieval driven by shared entities or surface overlap, not narrative structure.

Our structural representations hold only event triggers (verbs) and MAVEN types — no names — so lexical bias cannot live there. It lives in the baselines (BoW, TF-IDF, raw text), which *beat* our structural models. This cell shows the baselines win by matching shared names, not plot.

**Method:**
1. **Detect entities** — run spaCy NER over each summary; tag `PERSON`, `GPE`/`LOC` (places), `ORG`, `DATE` spans.
2. **Ablate** — for every query a baseline got right at rank 1, delete those entity tokens from its vectors and re-run retrieval.
3. **Measure** — count how many correct matches collapse. A big drop ⇒ the baseline was retrieving on shared names, not narrative.
4. **Contrast** — those same queries already failed under our name-free structural model ⇒ the baseline's edge was purely the entity shortcut structure discards.

Mirrors Hatzel & Biemann (2024): removing names drops StoryEmbed P@N 85.90 → 65.89.

In [ ]:
# Backup if Givanni's ideas are not working, taking too long. Not a priotiy. Prior work has sad this, so no need to compute yourslef. 

## (ii) Structural Alignment

**Goal:** Two summaries use different words but tell the same story, and the structural model still finds the match.

This is the good case we want. The two summaries barely share any words, so the baselines (which match on shared words) fail to connect them. But our structural model still retrieves the right story — because both summaries have the same chain of events. This is where throwing away the words and keeping only the events *helps*: the events reveal the plot that the different wording was hiding. It is the exact opposite of lexical bias: there the baseline won on shared names; here the structural model wins with no shared words at all.

**Method:**
1. **Find candidates** — queries where a structural condition (events_only / temporal / causal / joint) retrieves the right story at rank 1, but BoW/TF-IDF similarity to that story is low (ideally the lexical baseline missed it).
2. **Confirm vocabulary differs** — few shared content words between the two summaries, so the match is not coming from surface words.
3. **Show the events line up** — the two summaries share a similar event-type sequence / relation pattern; that is what carried the match.
4. **Genuine structure win** — structure succeeds exactly where surface overlap can't.

Given the aggregate result (structure *loses*), expect these cases to be rare — their count vs. the lexical-bias cases is the real story for Table 5.

In [ ]:
# In the categories level, it should wokr. This would still make sense, maybe there are there are many stories that follow similar pattern, and that's why we cannot discriminate uniquer stories. Maybe there's a prototypical narrative structure that match exactly.. 
# that's shy we might have bad results --

## (iii) Structural Mismatch

**Goal:** Adding temporal/causal relations makes two unrelated stories look alike, so the model retrieves the wrong one.

This is the bad case caused by structure itself. We have events that retrieve the right story fine. Then we add the relations (`BEFORE`, `CAUSE`, etc.) on top — and retrieval gets worse. Why? Because almost every story ends up with the same generic relation skeleton (mostly `BEFORE`/`CONTAINS`/`CAUSE` chains), so the relations make unrelated stories look similar instead of telling them apart. The relation layer adds *sameness, not signal* — and that false similarity drags the wrong story to the top.

**Method:**
1. **Find candidates** — queries where events_only retrieved the right story near the top, but adding a relation condition (temporal / causal / joint) pushed it down and put a wrong story at rank 1.
2. **Look at the wrong match** — it shares the same generic relation pattern (lots of `BEFORE`/`CONTAINS`, or `CAUSE_BEFORE`) as the query, even though the actual events differ.
3. **Show the relations are boilerplate** — relation labels are low-diversity and near-identical across unrelated stories, so they add noise, not discrimination.
4. **Structure actively hurt** — the opposite of (ii): here the relations created a false match and broke a retrieval that events alone got right.

This is the mechanism behind the negative RQ2/RQ3 result: relations are too generic to discriminate, so enriching with them degrades retrieval.

## (iv) Extraction Error

**Goal:** The events themselves are wrong — missing, noisy, or mislabeled — so the structural representation is broken *before* retrieval even starts.

Unlike (iii), the problem here isn't the relations — it's the **events** BERT+CRF pulled out of the text. If the events are junk, everything built on top (relations, linearized string, embedding) is junk too. Three kinds of bad event:
- **missing** — a real event the model never detected;
- **noisy** — a word tagged as an event that isn't a story event (e.g. the linking verbs `causes`, `makes`, `due`);
- **misclassified** — a real event given the wrong MAVEN type.

**What the pipeline already removed (so we don't re-count it here):**
- **Hallucinated relation IDs** — Llama inventing links to events that don't exist — were dropped in §4.6 and saved to `hallucinated_relations.jsonl`. The pattern: ~98% invent exactly `e(N+1)`, almost always linking the *last* real event — the model assumes everything must connect to something (UNI-60).
- **Broken / unparseable relations** — JSON parse failures and context-overflow summaries — were dropped whole in §4.5 (complete-case), with counts in `experiment.yaml`.

So the live work here is the **event-level noise that survived** because it looks valid.

**Method:**
1. **Count noisy triggers** — fraction of events whose trigger is a relational/linking verb (`causes`, `makes`, `enables`, `due`) or a subword fragment (`cing`); these are extraction artifacts, not story events (UNI-68; subword slicing was fixed in UNI-105 — verify residual ≈ 0).
2. **Spot misclassification** — sample events and check trigger vs assigned MAVEN type for obvious mismatches.
3. **Report the noise budget** — pull the hallucination rate from `hallucinated_relations.jsonl` and the parse/overflow drop counts from `experiment.yaml`, so structural noise is quantified end-to-end.
4. **Tie to retrieval** — on the worst-degrading queries (largest rank drop under structure vs baseline, UNI-103), inspect the events: is the failure explained by junk triggers rather than by the method itself?

## (v) Estimating Extraction & Relation Quality (No Gold Labels)

Tell Me Again! has no ground-truth event or temporal/causal annotations, so we cannot compute true accuracy. That is normal in NLP — we defend quality with a small-sample manual precision estimate, exactly as the MAVEN authors did for their own dataset.

**Method:**
1. Randomly sample 50 summaries from the test set.
2. Read each and mark by hand: (a) extracted events — is the trigger a real event, and is the MAVEN type right? (b) inferred relations — does each link actually make sense?
3. Report estimated event precision and estimated relation precision (% correct).

**Literature:** MAVEN validated its own labels the same way: *"One of the authors also manually examined 50 random documents. The estimated accuracies of event type annotation and event mention merging are 90.1% and 86.0% respectively"* (Wang et al., 2020). We mirror this to report an *estimated relation precision* for the LLM-inferred temporal/causal links.
